# IMDb dataset: data quality and exploration (PySpark)


## 1. Start Spark

`MAX_ROWS` was created for testing:

- `MAX_ROWS = 400` (test mode) reads only the first 400 rows of each IMDb file. The whole notebook runs in a few minutes, but the results are not meaningful: most joins find few or no matching rows.
- `MAX_ROWS = -1` (full run) reads every row of every file.


In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pyspark.sql import SparkSession, functions as F

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

DATA_DIR = Path("imdb_data")
DATA_DIR.mkdir(exist_ok=True)
SPARK_DATA_DIR = Path(os.environ.get("SPARK_DATA_DIR", "spark_data"))

MAX_ROWS = int(os.environ.get("MAX_ROWS", "400"))
RUN_NAME = "full" if MAX_ROWS < 0 else f"sample_{MAX_ROWS}"
WORK_DIR = SPARK_DATA_DIR / RUN_NAME
OUTPUT_DIR = Path("results") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPARK_DRIVER_MEMORY = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

FEATURE_MIN_RUNTIME = 70
SUCCESS_RATING = 7.5
MIN_SUCCESS_VOTES = 1000
CURRENT_YEAR = pd.Timestamp.today().year

spark = (
    SparkSession.builder
    .appName("imdb-data-quality-and-exploration")
    .master("local[*]")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Run mode: {RUN_NAME} (MAX_ROWS = {MAX_ROWS}), results in {OUTPUT_DIR}/")


def scalar(sql):
    return spark.sql(sql).first()[0]


def export(df, filename, n=100):
    pdf = df.toPandas()
    display(pdf.head(n))
    pdf.to_csv(OUTPUT_DIR / filename, index=False)
    return pdf


def save_table(name, sql):
    path = str(WORK_DIR / "clean" / name)
    spark.sql(sql).write.mode("overwrite").parquet(path)
    spark.read.parquet(path).createOrReplaceTempView(name)


spark

## 2. Load the files into Spark

In [ ]:
TABLE_FILES = {
    "title_basics": "title.basics.tsv.gz",
    "title_ratings": "title.ratings.tsv.gz",
    "title_akas": "title.akas.tsv.gz",
    "title_crew": "title.crew.tsv.gz",
    "title_episode": "title.episode.tsv.gz",
    "title_principals": "title.principals.tsv.gz",
    "name_basics": "name.basics.tsv.gz",
}

for table, filename in TABLE_FILES.items():
    target = WORK_DIR / "raw" / table
    if not (target / "_SUCCESS").exists():
        print(f"Converting {filename} to Parquet ...")
        raw = (spark.read
            .options(sep="\t", header=True, nullValue="\\N", quote="", inferSchema=False)
            .csv(str(DATA_DIR / filename)))
        if MAX_ROWS > 0:
            raw = raw.limit(MAX_ROWS)
        raw.write.mode("overwrite").parquet(str(target))
    spark.read.parquet(str(target)).createOrReplaceTempView(table)

## 3. verify files have been loaded

In [ ]:
raw_counts = {table: spark.table(table).count() for table in TABLE_FILES}
for table, count in raw_counts.items():
    print(f"{table:18s} {count:>12,} rows")

# Data preparation

## 4. Percentage of null, missing or empty values per column

IMDb uses `\N` for missing values; the loader turns this marker into NULL.

In [ ]:
null_rows = []

for table in TABLE_FILES:
    df = spark.table(table)
    counters = [F.count(F.lit(1)).alias("__total")] + [
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]
    row = df.agg(*counters).first().asDict()
    total = row["__total"]
    for c in df.columns:
        null_rows.append((table, c, total, row[c], round(100 * row[c] / total, 2) if total else None))

# let's export the percentage of null/missing/empty values for every column of every table
null_report = export(
    spark.createDataFrame(null_rows, "table string, column string, total_rows long, missing_rows long, missing_pct double")
    .orderBy(F.desc("missing_pct"), "table", "column"),
    "imdb_null_percentage_by_column.csv",
)

### Five columns with the highest percentage of null values

In [ ]:
top_5_missing = null_report.nlargest(5, "missing_pct") 
display(top_5_missing)
top_5_missing.to_csv(OUTPUT_DIR / "top_5_missing_columns.csv", index=False)

## 5. Other data-quality issues

The checks cover impossible or future dates, people credited on titles released before their birth, duplicate primary keys, out-of-range values, inconsistent series metadata, and foreign keys whose parent record is missing

In [ ]:
quality_checks = []

def add_check(category, issue, sql):
    quality_checks.append((category, issue, scalar(sql)))

add_check("Impossible dates", "People with deathYear earlier than birthYear", '''
    SELECT COUNT(*) FROM name_basics
    WHERE CAST(deathYear AS INT) < CAST(birthYear AS INT)
''')
add_check("Impossible dates", "Principals who died before they were born", '''
    SELECT COUNT(DISTINCT p.nconst) FROM title_principals p
    JOIN name_basics n ON n.nconst = p.nconst
    WHERE CAST(n.deathYear AS INT) < CAST(n.birthYear AS INT)
''')
add_check("Impossible dates", f"People with birthYear or deathYear after {CURRENT_YEAR}", f'''
    SELECT COUNT(*) FROM name_basics
    WHERE CAST(birthYear AS INT) > {CURRENT_YEAR} OR CAST(deathYear AS INT) > {CURRENT_YEAR}
''')
add_check("Impossible dates", "Principal credits on titles released before the person's birth (archive footage excluded)", '''
    SELECT COUNT(*) FROM title_principals p
    JOIN name_basics n ON n.nconst = p.nconst
    JOIN title_basics b ON b.tconst = p.tconst
    WHERE p.category NOT IN ('archive_footage', 'archive_sound')
      AND CAST(b.startYear AS INT) < CAST(n.birthYear AS INT)
''')
add_check("Impossible dates", "Titles with endYear earlier than startYear", '''
    SELECT COUNT(*) FROM title_basics
    WHERE CAST(endYear AS INT) < CAST(startYear AS INT)
''')
add_check("Duplicates", "Duplicate title IDs in title_basics", '''
    SELECT COUNT(*) FROM (SELECT tconst FROM title_basics GROUP BY tconst HAVING COUNT(*) > 1) d
''')
add_check("Duplicates", "Duplicate person IDs in name_basics", '''
    SELECT COUNT(*) FROM (SELECT nconst FROM name_basics GROUP BY nconst HAVING COUNT(*) > 1) d
''')
add_check("Invalid values", "Ratings outside 1-10 or not numeric", '''
    SELECT COUNT(*) FROM title_ratings
    WHERE CAST(averageRating AS DOUBLE) IS NULL OR CAST(averageRating AS DOUBLE) NOT BETWEEN 1 AND 10
''')
add_check("Invalid values", "Ratings with negative or non-numeric vote counts", '''
    SELECT COUNT(*) FROM title_ratings
    WHERE CAST(numVotes AS BIGINT) IS NULL OR CAST(numVotes AS BIGINT) < 0
''')
add_check("Invalid values", "Titles with non-positive or non-numeric runtime", '''
    SELECT COUNT(*) FROM title_basics
    WHERE runtimeMinutes IS NOT NULL AND COALESCE(CAST(runtimeMinutes AS INT), 0) <= 0
''')
add_check("Invalid values", "Titles with isAdult not in {0, 1}", '''
    SELECT COUNT(*) FROM title_basics
    WHERE isAdult IS NULL OR isAdult NOT IN ('0', '1')
''')
add_check("Invalid values", "Episodes with non-positive season/episode numbers", '''
    SELECT COUNT(*) FROM title_episode
    WHERE CAST(seasonNumber AS INT) <= 0 OR CAST(episodeNumber AS INT) <= 0
''')
add_check("Inconsistent metadata", "Episodes whose parent is not a TV series or mini-series", '''
    SELECT COUNT(*) FROM title_episode e
    JOIN title_basics b ON b.tconst = e.parentTconst
    WHERE b.titleType NOT IN ('tvSeries', 'tvMiniSeries')
''')
add_check("Missing parent record", "Ratings referencing missing titles", '''
    SELECT COUNT(*) FROM title_ratings r LEFT ANTI JOIN title_basics b ON b.tconst = r.tconst
''')
add_check("Missing parent record", "Alternative titles referencing missing titles", '''
    SELECT COUNT(*) FROM title_akas a LEFT ANTI JOIN title_basics b ON b.tconst = a.titleId
''')
add_check("Missing parent record", "Crew rows referencing missing titles", '''
    SELECT COUNT(*) FROM title_crew c LEFT ANTI JOIN title_basics b ON b.tconst = c.tconst
''')
add_check("Missing parent record", "Director credits referencing missing people", '''
    SELECT COUNT(*) FROM (
        SELECT explode(split(directors, ',')) AS nconst FROM title_crew WHERE directors IS NOT NULL
    ) d LEFT ANTI JOIN name_basics n ON n.nconst = d.nconst
''')
add_check("Missing parent record", "Writer credits referencing missing people", '''
    SELECT COUNT(*) FROM (
        SELECT explode(split(writers, ',')) AS nconst FROM title_crew WHERE writers IS NOT NULL
    ) w LEFT ANTI JOIN name_basics n ON n.nconst = w.nconst
''')
add_check("Missing parent record", "Episode rows referencing missing episode titles", '''
    SELECT COUNT(*) FROM title_episode e LEFT ANTI JOIN title_basics b ON b.tconst = e.tconst
''')
add_check("Missing parent record", "Episodes referencing missing parent series", '''
    SELECT COUNT(*) FROM title_episode e LEFT ANTI JOIN title_basics b ON b.tconst = e.parentTconst
''')
add_check("Missing parent record", "Principals referencing missing titles", '''
    SELECT COUNT(*) FROM title_principals p LEFT ANTI JOIN title_basics b ON b.tconst = p.tconst
''')
add_check("Missing parent record", "Principals referencing missing people", '''
    SELECT COUNT(*) FROM title_principals p LEFT ANTI JOIN name_basics n ON n.nconst = p.nconst
''')
add_check("Missing parent record", "knownForTitles entries referencing missing titles", '''
    SELECT COUNT(*) FROM (
        SELECT explode(split(knownForTitles, ',')) AS tconst FROM name_basics WHERE knownForTitles IS NOT NULL
    ) k LEFT ANTI JOIN title_basics b ON b.tconst = k.tconst
''')

quality_report = export(  # Data-quality issues found in the dataset and number of affected rows
    spark.createDataFrame(quality_checks, "category string, issue string, affected_rows long")
    .orderBy("category", F.desc("affected_rows")),
    "imdb_data_quality_issues.csv",
)

### Examples of the issues

In [ ]:
principals_died_before_born = spark.sql( 
    '''
    SELECT DISTINCT n.nconst, n.primaryName, n.birthYear, n.deathYear
    FROM title_principals p
    JOIN name_basics n ON n.nconst = p.nconst
    WHERE CAST(n.deathYear AS INT) < CAST(n.birthYear AS INT)
    ORDER BY n.primaryName
''').toPandas()
display(principals_died_before_born)

display(spark.sql('''
    SELECT e.* FROM title_episode e
    LEFT ANTI JOIN title_basics b ON b.tconst = e.parentTconst
''').limit(20).toPandas())

display(spark.sql('''
    SELECT p.tconst, b.primaryTitle, b.startYear, p.nconst, n.primaryName, n.birthYear, p.category
    FROM title_principals p
    JOIN name_basics n ON n.nconst = p.nconst
    JOIN title_basics b ON b.tconst = p.tconst
    WHERE p.category NOT IN ('archive_footage', 'archive_sound')
      AND CAST(b.startYear AS INT) < CAST(n.birthYear AS INT)
''').limit(20).toPandas())

## 6. Handling the data-quality issues

The raw tables are left untouched; cleaned tables are written to Parquet with Spark and used by every analysis below.

| Issue | Handling | Reason |
|---|---|---|
| `\N` / blank strings | Stored as NULL | One consistent representation of "unknown" |
| Non-numeric numbers | `CAST` → NULL | Bad values do not stop the pipeline nor become fake zeros |
| deathYear < birthYear, future birth/death years | Both years set to NULL, `date_issue_flag = true`; the person is kept | We cannot know which year is wrong; dropping the person would orphan all their credits |
| Credits before birth | Kept, flagged with `credit_before_birth` | Often a wrong birthYear; credits are still valid |
| endYear < startYear | endYear set to NULL | startYear is the field used in analyses |
| runtime ≤ 0 | NULL | Excluded from runtime statistics |
| isAdult not 0/1 | NULL | Unknown flag rather than a guess |
| Rating outside 1–10, negative votes | Row removed from `ratings_clean` | Cannot be used in rating analyses |
| Season/episode ≤ 0 | NULL | Episode kept, season unknown |
| Duplicate primary keys | One row kept per key | Keys must be unique for joins |
| Orphan foreign keys | Rows removed from the cleaned child table (`LEFT SEMI JOIN` on the parent; counts kept in the report above) | Inner joins would drop them anyway; this makes it explicit and auditable |

Missing values are never imputed: inventing IDs, dates, ratings or runtimes would fabricate facts.

In [ ]:
DATE_ISSUE = f"COALESCE(d < b OR b > {CURRENT_YEAR} OR d > {CURRENT_YEAR}, false)"

save_table("titles_clean", '''
WITH typed AS (
    SELECT tconst, titleType, primaryTitle, originalTitle, isAdult,
           CAST(startYear AS INT) AS startYear,
           CAST(endYear AS INT) AS endYear,
           CAST(runtimeMinutes AS INT) AS runtimeMinutes,
           NULLIF(TRIM(genres), '') AS genres,
           ROW_NUMBER() OVER (PARTITION BY tconst ORDER BY primaryTitle) AS rn
    FROM title_basics
    WHERE tconst IS NOT NULL AND TRIM(tconst) <> ''
)
SELECT tconst, titleType, primaryTitle, originalTitle,
       CASE WHEN isAdult IN ('0', '1') THEN CAST(isAdult AS INT) END AS isAdult,
       startYear,
       CASE WHEN endYear >= startYear OR startYear IS NULL THEN endYear END AS endYear,
       CASE WHEN runtimeMinutes > 0 THEN runtimeMinutes END AS runtimeMinutes,
       genres
FROM typed
WHERE rn = 1
''')

save_table("people_clean", f'''
WITH typed AS (
    SELECT nconst, primaryName, primaryProfession, knownForTitles,
           CAST(birthYear AS INT) AS b, CAST(deathYear AS INT) AS d,
           ROW_NUMBER() OVER (PARTITION BY nconst ORDER BY primaryName) AS rn
    FROM name_basics
    WHERE nconst IS NOT NULL AND TRIM(nconst) <> ''
)
SELECT nconst, primaryName,
       CASE WHEN {DATE_ISSUE} THEN NULL ELSE b END AS birthYear,
       CASE WHEN {DATE_ISSUE} THEN NULL ELSE d END AS deathYear,
       {DATE_ISSUE} AS date_issue_flag,
       primaryProfession, knownForTitles
FROM typed
WHERE rn = 1
''')

save_table("ratings_clean", '''
WITH typed AS (
    SELECT r.tconst,
           CAST(r.averageRating AS DOUBLE) AS averageRating,
           CAST(r.numVotes AS BIGINT) AS numVotes,
           ROW_NUMBER() OVER (PARTITION BY r.tconst ORDER BY CAST(r.numVotes AS BIGINT) DESC) AS rn
    FROM title_ratings r
    LEFT SEMI JOIN titles_clean t ON t.tconst = r.tconst
)
SELECT tconst, averageRating, numVotes
FROM typed
WHERE rn = 1 AND averageRating BETWEEN 1 AND 10 AND numVotes >= 0
''')

save_table("principals_clean", '''
SELECT p.tconst, CAST(p.ordering AS INT) AS ordering, p.nconst, p.category, p.job, p.characters,
       COALESCE(t.startYear < n.birthYear AND p.category NOT IN ('archive_footage', 'archive_sound'), false)
           AS credit_before_birth
FROM title_principals p
JOIN titles_clean t ON t.tconst = p.tconst
JOIN people_clean n ON n.nconst = p.nconst
''')

for role, column in [("directors_clean", "directors"), ("writers_clean", "writers")]:
    save_table(role, f'''
    SELECT DISTINCT c.tconst, c.nconst
    FROM (
        SELECT tconst, explode(split({column}, ',')) AS nconst
        FROM title_crew WHERE {column} IS NOT NULL
    ) c
    LEFT SEMI JOIN titles_clean t ON t.tconst = c.tconst
    LEFT SEMI JOIN people_clean p ON p.nconst = c.nconst
    ''')

save_table("episodes_clean", '''
SELECT e.tconst, e.parentTconst,
       CASE WHEN CAST(e.seasonNumber AS INT) > 0 THEN CAST(e.seasonNumber AS INT) END AS seasonNumber,
       CASE WHEN CAST(e.episodeNumber AS INT) > 0 THEN CAST(e.episodeNumber AS INT) END AS episodeNumber
FROM title_episode e
LEFT SEMI JOIN titles_clean t ON t.tconst = e.tconst
LEFT SEMI JOIN titles_clean s ON s.tconst = e.parentTconst
''')

save_table("akas_clean", '''
SELECT a.titleId AS tconst, CAST(a.ordering AS INT) AS ordering, a.title,
       NULLIF(TRIM(a.region), '') AS region, NULLIF(LOWER(TRIM(a.language)), '') AS language,
       a.types, a.attributes, a.isOriginalTitle = '1' AS isOriginalTitle
FROM title_akas a
LEFT SEMI JOIN titles_clean t ON t.tconst = a.titleId
''')

CLEAN_TABLES = {
    "title_basics": "titles_clean", "name_basics": "people_clean", "title_ratings": "ratings_clean",
    "title_principals": "principals_clean", "title_episode": "episodes_clean", "title_akas": "akas_clean",
}
clean_counts = {clean: spark.table(clean).count() for clean in CLEAN_TABLES.values()}
cleaning_summary = pd.DataFrame([  # Row counts before and after handling the data-quality issues
    {"raw_table": raw, "clean_table": clean, "raw_rows": raw_counts[raw],
     "clean_rows": clean_counts[clean], "removed_rows": raw_counts[raw] - clean_counts[clean]}
    for raw, clean in CLEAN_TABLES.items()
])
display(cleaning_summary)
cleaning_summary.to_csv(OUTPUT_DIR / "imdb_cleaning_summary.csv", index=False)

### Verification: the issues are no longer present in the cleaned tables

In [ ]:
post_cleaning_checks = pd.DataFrame([  # Remaining number of problematic rows after cleaning (all expected to be 0)
    {"check": "deathYear < birthYear", "remaining_rows": scalar("SELECT COUNT(*) FROM people_clean WHERE deathYear < birthYear")},
    {"check": "endYear < startYear", "remaining_rows": scalar("SELECT COUNT(*) FROM titles_clean WHERE endYear < startYear")},
    {"check": "runtime <= 0", "remaining_rows": scalar("SELECT COUNT(*) FROM titles_clean WHERE runtimeMinutes <= 0")},
    {"check": "rating outside 1-10", "remaining_rows": scalar("SELECT COUNT(*) FROM ratings_clean WHERE averageRating NOT BETWEEN 1 AND 10")},
    {"check": "duplicate tconst", "remaining_rows": scalar("SELECT COUNT(*) - COUNT(DISTINCT tconst) FROM titles_clean")},
    {"check": "duplicate nconst", "remaining_rows": scalar("SELECT COUNT(*) - COUNT(DISTINCT nconst) FROM people_clean")},
    {"check": "principals without title", "remaining_rows": scalar(
        "SELECT COUNT(*) FROM principals_clean p LEFT ANTI JOIN titles_clean t ON t.tconst = p.tconst")},
    {"check": "principals without person", "remaining_rows": scalar(
        "SELECT COUNT(*) FROM principals_clean p LEFT ANTI JOIN people_clean n ON n.nconst = p.nconst")},
    {"check": "episodes without parent", "remaining_rows": scalar(
        "SELECT COUNT(*) FROM episodes_clean e LEFT ANTI JOIN titles_clean t ON t.tconst = e.parentTconst")},
    {"check": "season/episode <= 0", "remaining_rows": scalar(
        "SELECT COUNT(*) FROM episodes_clean WHERE seasonNumber <= 0 OR episodeNumber <= 0")},
])
display(post_cleaning_checks)